# Philo on Hakhel using Perseus MCP

This notebook investigates passages in Philo of Alexandria which may be related to Deuteronomy 31:10–13: the seventh-year assembly at the Feast of Booths, the public hearing of the law, and the presence of the whole community.

The aim is fairly modest. We will use Perseus MCP to discover the available tools, search the Greek corpus, retrieve the full passages behind promising snippets, and export an evidence table that can be checked by CTS URN. The keyword searches are meant to lead us to texts for close reading; a hit is not in itself proof that Philo is discussing Hakhel, and a missing hit is certainly not proof that he is silent about it.

## Table of content (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Research question and limits</a>
* <a href="#bullet2">2 - Install the MCP server and notebook dependencies</a>
* <a href="#bullet3">3 - Load Perseus MCP</a>
* <a href="#bullet4">4 - Handle MCP responses</a>
* <a href="#bullet5">5 - Inspect the live Perseus MCP tool registry</a>
* <a href="#bullet6">6 - Verify the Philo scope</a>
* <a href="#bullet7">7 - Build a subject-oriented search plan</a>
    * <a href="#bullet7x1">7.1 - Search-index caution: lemma vs. form</a>
* <a href="#bullet8">8 - Search Philo through Perseus MCP</a>
* <a href="#bullet9">9 - Normalize and deduplicate the search evidence</a>
* <a href="#bullet10">10 - Retrieve full passage text through Perseus MCP</a>
* <a href="#bullet11">11 - Focus on likely Hakhel contexts</a>
* <a href="#bullet12">12 - Check the collected evidence</a>
* <a href="#bullet13">13 - Export reproducible results</a>
* <a href="#bullet14">14 - Questions for close reading</a>
* <a href="#bullet15">15 - Required libraries</a>
* <a href="#bullet16">16 - Notebook details</a>

# 1 - Research question and limits<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

The starting question is:

> How does Philo describe ideas connected with Hakhel—especially the seventh-year setting, the Feast of Booths, public reading or hearing of the law, and the participation of the community?

Deuteronomy 31:10–13 itself is the historical and literary starting point, but this notebook searches **Philo's Greek corpus**, not the biblical text. Greek search terms are therefore probes, not translations guaranteed to reproduce Philo's wording.

A few points should be kept in mind:

- a keyword hit is evidence for inspection, not proof of conceptual relevance;
- absence from a search result is not proof that Philo does not discuss the subject;
- search indexing, tokenization, lemmatization, and available editions affect recall;
- interpretation must be based on retrieved passage context and cited CTS URNs.

# 2 - Install the MCP server and notebook dependencies<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

`perseus-mcp` is a local MCP server. The notebook imports its registered FastMCP server object and connects to it in-process. An external desktop client normally launches the same server over stdio; the exposed tools and their arguments are the same.

Restart the kernel after installation if imports fail in an already-running environment.

In [1]:
# Install the MCP server and notebook dependencies
%pip install -q perseus-mcp pandas tqdm

# 3 - Load Perseus MCP<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

All corpus operations below go through `client.call_tool(...)`. Importing `server.mcp` does not bypass MCP tool boundaries; it supplies an in-process transport suitable for a self-contained notebook.

In [2]:
# Load Perseus MCP
from __future__ import annotations

import asyncio
import contextlib
import io
import logging
import html
import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm
from fastmcp import Client
from perseus_mcp import server

mcp = server.mcp
OUTPUT_DIR = Path("philo_hakhel_perseus_results")
OUTPUT_DIR.mkdir(exist_ok=True)

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

print(f"Loaded MCP server: {mcp.name}")
print(f"Output directory: {OUTPUT_DIR.relative_to(REPOSITORY_DIR).as_posix()}")

Loaded MCP server: perseus
Output directory: philo/philo_hakhel_perseus_results


# 4 - Handle MCP responses<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

Perseus MCP tools return text content blocks. Some contain JSON and others contain plain text or XML. These helpers preserve that distinction.

In [3]:
# Handle MCP responses
def tool_text(result: Any) -> str:
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_text(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> str:
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


async def call_json(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> Any:
    return json.loads(await call_text(client, tool_name, arguments))


TAG_RE = re.compile(r"<[^>]+>")


def clean_text(value: Any) -> str:
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)
    return " ".join(html.unescape(TAG_RE.sub("", str(value or ""))).split())

# 5 - Inspect the live Perseus MCP tool registry<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

We first ask the server which tools it actually offers, rather than relying on remembered names or an older schema. The small check below confirms that the tools needed in the rest of this notebook are present.

In [4]:
# Inspect the live Perseus MCP tool registry
async with Client(mcp) as client:
    tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in tools}
required_tools = {
    "get_scaife_library_metadata",
    "search_perseus",
    "get_scaife_passage_text",
}
missing = required_tools - set(tool_by_name)
assert not missing, f"Required Perseus MCP tools are missing: {sorted(missing)}"

pd.DataFrame(
    [
        {
            "name": tool.name,
            "description": (tool.description or "").splitlines()[0],
        }
        for tool in tools
    ]
)

,name,description
0,get_passage,Get the text of a specific passage using a CTS...
1,get_passage_plus,Get passage text plus surrounding metadata/con...
2,get_passage_plaintext,Get a passage as plain readable text instead o...
3,get_valid_references,"Get valid citations/references for a work, use..."
4,get_valid_references_json,Get valid citation references as paged JSON in...
5,count_valid_references,Count valid citation references without return...
6,get_capabilities,Get the list of available texts and editions f...
7,get_cache_status,Get local metadata cache status.
8,refresh_metadata_cache,Refresh cached CTS and Scaife library metadata.
9,clear_metadata_cache,Clear local metadata cache files and in-memory...


# 6 - Verify the Philo scope<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

Philo is represented in the Scaife library used by Perseus MCP as textgroup:

`urn:cts:greekLit:tlg0018`

One technical point is worth noting: the legacy Perseus CTS capabilities inventory and the Scaife library inventory are not identical. A CTS author lookup may return no Philo record even though Scaife contains Philo's Greek works. Because the search and passage evidence in this notebook comes from the Scaife-backed Perseus MCP tools, the Scaife metadata check is the one we will use here.

In [5]:
# Verify the Philo scope
PHILO_TEXTGROUP = "urn:cts:greekLit:tlg0018"

async with Client(mcp) as client:
    philo_metadata = await call_json(
        client,
        "get_scaife_library_metadata",
        {"urn": PHILO_TEXTGROUP},
    )

assert philo_metadata.get("urn") == PHILO_TEXTGROUP

scope_summary = {
    "urn": philo_metadata.get("urn"),
    "label": philo_metadata.get("label") or philo_metadata.get("title") or philo_metadata.get("name"),
    "work_count": len(philo_metadata.get("works") or []),
}
print(json.dumps(scope_summary, ensure_ascii=False, indent=2))

{
  "urn": "urn:cts:greekLit:tlg0018",
  "label": "Philo Judaeus",
  "work_count": 31
}


# 7 - Build a subject-oriented search plan<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

The terms cover separate parts of the Hakhel question:

- `σκηνα*`: tents, booths, or tabernacle language;
- `ἑβδομά*`: seven, a group of seven, or seventh-period language;
- `ἀνάγνω*`: reading (which may also produce irrelevant philological uses);
- `ἀκρο*`: hearing or attentive listening;
- `νόμ*`: law;
- `παιδ*` and `τέκν*`: children or descendants;
- `γυναικ*` and `ἀνδρ*`: women and men, for audience-formula checks.

Each search is explicitly restricted to Philo's textgroup. The notebook collects only a small number of pages because its purpose is a basic and inspectable first survey rather than an exhaustive concordance. These are **surface-form probes**, not dictionary lemmata; the wildcard terms were chosen for a practical reason: Philo's Scaife texts return form hits but not reliable lemma hits.

In [6]:
# Build a subject-oriented search plan
SEARCH_TERMS = [
    ("feast_of_booths", "σκηνα*", "form"),
    ("seven", "ἑβδομά*", "form"),
    ("reading", "ἀνάγνω*", "form"),
    ("hearing", "ἀκρο*", "form"),
    ("law", "νόμ*", "form"),
    ("child", "παιδ*", "form"),
    ("offspring", "τέκν*", "form"),
    ("women", "γυναικ*", "form"),
    ("men", "ἀνδρ*", "form"),
]
MAX_PAGES_PER_TERM = 2

pd.DataFrame(SEARCH_TERMS, columns=["theme", "query", "search_kind"])

,theme,query,search_kind
0,feast_of_booths,σκηνα*,form
1,seven,ἑβδομά*,form
2,reading,ἀνάγνω*,form
3,hearing,ἀκρο*,form
4,law,νόμ*,form
5,child,παιδ*,form
6,offspring,τέκν*,form
7,women,γυναικ*,form
8,men,ἀνδρ*,form


## 7.1 - Search-index caution: lemma vs. form<a class="anchor" id="bullet7x1"></a>
##### [Back to ToC](#TOC)

Perseus/Scaife exposes two relevant search modes. `search_kind="form"` searches surface forms in the text, while `search_kind="lemma"` searches a lemmatized index when that index exists for the underlying dataset.

A zero-hit lemma search is therefore not automatically evidence that the concept or word is absent from Philo. In a diagnostic check, `νόμος` with `search_kind="lemma"` and `text_group="urn:cts:greekLit:tlg0018"` returned zero Philo hits, even though the same term with `search_kind="form"` returned Philo passages. The same lemma query also returns hits elsewhere in the broader Scaife corpus. The issue is the level of annotation/indexing available for the selected dataset, not the Greek word itself.

The amount of detail returned by Perseus MCP is dataset-dependent. Some texts have rich token/lemma/search metadata; others only expose reliable surface text, passage URNs, labels, and snippets. For Philo, form searches and passage retrieval by CTS URN are the safer discovery path; lemma searches should be treated as optional diagnostics rather than negative evidence.

This also affects recall for exact form searches. A query for a dictionary headword such as `σκηνοπηγία` only matches that surface spelling; inflected forms require additional forms, wildcards, or terms derived from confirmed passages.

# 8 - Search Philo through Perseus MCP<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

The pagination helper stops when the service reports the final page or returns no results. Live result counts and ordering can change, so no fixed hit count is asserted.

In [7]:
# Search Philo through Perseus MCP
MAX_SEARCH_ATTEMPTS = 3
SEARCH_RETRY_DELAY_SECONDS = 2


async def search_page_payload(
    client: Client,
    query: str,
    search_kind: str,
    page_num: int,
) -> dict[str, Any]:
    arguments = {
        "query": query,
        "language": "greek",
        "query_format": "unicode",
        "search_kind": search_kind,
        "page_num": page_num,
        "text_group": PHILO_TEXTGROUP,
        "result_format": "instances",
    }
    last_error = None
    for attempt in range(1, MAX_SEARCH_ATTEMPTS + 1):
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                payload = await call_json(client, "search_perseus", arguments)
            if not isinstance(payload, dict):
                return {
                    "results": [],
                    "total_count": None,
                    "page": {"number": page_num, "has_next": False},
                    "error": f"Non-dictionary search response: {type(payload).__name__}",
                    "attempts": attempt,
                }
            payload.setdefault("error", None)
            payload["attempts"] = attempt
            return payload
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_SEARCH_ATTEMPTS:
                await asyncio.sleep(SEARCH_RETRY_DELAY_SECONDS * attempt)

    return {
        "results": [],
        "total_count": None,
        "page": {"number": page_num, "has_next": False},
        "error": last_error,
        "attempts": MAX_SEARCH_ATTEMPTS,
    }


async def search_pages(client: Client, query: str, search_kind: str, max_pages: int) -> list[dict[str, Any]]:
    pages = []
    for page_num in range(1, max_pages + 1):
        payload = await search_page_payload(client, query, search_kind, page_num)
        pages.append(payload)
        if payload.get("error"):
            break
        results = payload.get("results") or []
        page = payload.get("page") or {}
        if not results or page.get("has_next") is False:
            break
    return pages


search_runs = {}
async with Client(mcp) as client:
    for theme, query, search_kind in tqdm(SEARCH_TERMS, desc="Searching Philo"):
        search_runs[theme] = await search_pages(
            client,
            query,
            search_kind,
            MAX_PAGES_PER_TERM,
        )

search_summary = []
for theme, query, search_kind in SEARCH_TERMS:
    pages = search_runs[theme]
    errors = [page.get("error") for page in pages if page.get("error")]
    search_summary.append(
        {
            "theme": theme,
            "query": query,
            "search_kind": search_kind,
            "reported_total": pages[0].get("total_count") if pages else None,
            "pages_collected": len(pages),
            "rows_collected": sum(len(page.get("results") or []) for page in pages),
            "failed_pages": len(errors),
            "error": "; ".join(errors) if errors else None,
        }
    )

pd.DataFrame(search_summary)

,theme,query,search_kind,reported_total,pages_collected,rows_collected,failed_pages,error
0,feast_of_booths,σκηνα*,form,15,2,15,0,None
1,seven,ἑβδομά*,form,112,2,20,0,None
2,reading,ἀνάγνω*,form,14,2,14,0,None
3,hearing,ἀκρο*,form,166,2,20,0,None
4,law,νόμ*,form,1328,2,20,0,None
5,child,παιδ*,form,521,2,20,0,None
6,offspring,τέκν*,form,121,2,20,0,None
7,women,γυναικ*,form,353,2,20,0,None
8,men,ἀνδρ*,form,674,2,20,0,None


# 9 - Normalize and deduplicate the search evidence<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

A passage can be found by more than one query. The normalized table keeps the URN, work label, snippet, and search theme, then deduplicates by passage URN plus theme. As a final check, the code stops if a result outside Philo slips into the table.

In [8]:
# Normalize and deduplicate the search evidence
rows = []
for theme, pages in search_runs.items():
    query = next(item[1] for item in SEARCH_TERMS if item[0] == theme)
    for page in pages:
        for item in page.get("results") or []:
            passage = item.get("passage") or {}
            text_info = passage.get("text") or {}
            ancestors = text_info.get("ancestors") or []
            work = ancestors[-1].get("label") if ancestors else text_info.get("label")
            rows.append(
                {
                    "theme": theme,
                    "query": query,
                    "urn": passage.get("urn"),
                    "work": work,
                    "human_reference": ((passage.get("refs") or {}).get("start") or {}).get("human_reference"),
                    "snippet": clean_text(item.get("content")),
                }
            )

hit_columns = ["theme", "query", "urn", "work", "human_reference", "snippet"]
hits_df = pd.DataFrame(rows, columns=hit_columns)
if not hits_df.empty:
    hits_df = hits_df.dropna(subset=["urn"]).drop_duplicates(subset=["urn", "theme"]).reset_index(drop=True)
    assert hits_df["urn"].str.startswith(PHILO_TEXTGROUP + ".").all(), "Out-of-scope search result detected"

print(f"Normalized {len(hits_df)} themed passage hits.")
hits_df.head(30)

Normalized 169 themed passage hits.


,theme,query,urn,work,human_reference,snippet
0,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:1.293,De Vita Mosis (Lib. I-II),Book 1 Section 293,"πόσον πλῆθος χρημάτων, ἀνοητότατε, καὶ δωρεῶν,..."
1,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:2.42,De Vita Mosis (Lib. I-II),Book 2 Section 42,μετὰ δὲ τὰς εὐχὰς καὶ τὰς εὐχαριστίας οἱ μὲν π...
2,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.207,De Specialibus Legibus (lib. i‑iv),Book 2 Section 207,"ὅτε μὲν γὰρ ἐν πεδίοις ἐστὶ τὰ τρέφοντα, μὴ συ..."
3,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg006.1st1K-grc1:98,De Posteritate Caini,Section 98,ταῦτα ἀποχρώντως λελέχθω περὶ τοῦ μεταλλοιοῦντ...
4,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.204,De Specialibus Legibus (lib. i‑iv),Book 2 Section 204,Τελευταία δ’ ἐστὶν ἑορτὴ τῶν ἐνιαυσίων προσαγο...
5,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:4.129,De Specialibus Legibus (lib. i‑iv),Book 4 Section 129,εἰκὸς μὲν οὖν τὸ τεράστιον τοῦ μεγαλουργηθέντο...
6,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.206,De Specialibus Legibus (lib. i‑iv),Book 2 Section 206,καὶ μὴν ἐν σκηναῖς προστέτακται διαιτᾶσθαι τὸν...
7,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg030.1st1K-grc1:116,In Flaccum,Section 116,"Τοῦτο καινότατον ὑπέμεινε Φλάκκος ἐν χώρᾳ, ἧς ..."
8,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg010.1st1K-grc1:31,De Plantatione,Section 31,τὰ δὲ ἀμείνω καὶ τελειότερα τῷ μεσαιτάτῳ καὶ κ...
9,feast_of_booths,σκηνα*,urn:cts:greekLit:tlg0018.tlg017.1st1K-grc1:153,De Fuga Et Inventione,Section 153,"ζητῶν οὖν 1 αὐτῶ H1 βούλετ’ H τῶν Mang.: τῶ G,..."


# 10 - Retrieve full passage text through Perseus MCP<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

Search snippets are useful for discovery but are not sufficient evidence for interpretation. This step retrieves the full text of each unique passage URN with `get_scaife_passage_text`.

In [9]:
# Retrieve full passage text through Perseus MCP
unique_urns = [] if hits_df.empty else hits_df["urn"].drop_duplicates().tolist()
passage_rows = []

async with Client(mcp) as client:
    for urn in tqdm(unique_urns, desc="Fetching passages"):
        try:
            text = await call_text(client, "get_scaife_passage_text", {"urn": urn})
            passage_rows.append({"urn": urn, "passage_text": clean_text(text), "error": None})
        except Exception as exc:
            passage_rows.append({"urn": urn, "passage_text": "", "error": repr(exc)})

passages_df = pd.DataFrame(passage_rows, columns=["urn", "passage_text", "error"])
evidence_df = hits_df.merge(passages_df, on="urn", how="left")

display_columns = ["theme", "work", "human_reference", "urn", "snippet", "passage_text", "error"]
evidence_df[display_columns].head(30) if not evidence_df.empty else evidence_df

,theme,work,human_reference,urn,snippet,passage_text,error
0,feast_of_booths,De Vita Mosis (Lib. I-II),Book 1 Section 293,urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:1.293,"πόσον πλῆθος χρημάτων, ἀνοητότατε, καὶ δωρεῶν,...","πόσον πλῆθος χρημάτων, ἀνοητότατε, καὶ δωρεῶν,...",None
1,feast_of_booths,De Vita Mosis (Lib. I-II),Book 2 Section 42,urn:cts:greekLit:tlg0018.tlg022.1st1K-grc1:2.42,μετὰ δὲ τὰς εὐχὰς καὶ τὰς εὐχαριστίας οἱ μὲν π...,μετὰ δὲ τὰς εὐχὰς καὶ τὰς εὐχαριστίας οἱ μὲν π...,None
2,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 207,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.207,"ὅτε μὲν γὰρ ἐν πεδίοις ἐστὶ τὰ τρέφοντα, μὴ συ...","ὅτε μὲν γὰρ ἐν πεδίοις ἐστὶ τὰ τρέφοντα, μὴ συ...",None
3,feast_of_booths,De Posteritate Caini,Section 98,urn:cts:greekLit:tlg0018.tlg006.1st1K-grc1:98,ταῦτα ἀποχρώντως λελέχθω περὶ τοῦ μεταλλοιοῦντ...,ταῦτα ἀποχρώντως λελέχθω περὶ τοῦ μεταλλοιοῦντ...,None
4,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 204,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.204,Τελευταία δ’ ἐστὶν ἑορτὴ τῶν ἐνιαυσίων προσαγο...,Τελευταία δ’ ἐστὶν ἑορτὴ τῶν ἐνιαυσίων προσαγο...,None
5,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 4 Section 129,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:4.129,εἰκὸς μὲν οὖν τὸ τεράστιον τοῦ μεγαλουργηθέντο...,εἰκὸς μὲν οὖν τὸ τεράστιον τοῦ μεγαλουργηθέντο...,None
6,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 206,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.206,καὶ μὴν ἐν σκηναῖς προστέτακται διαιτᾶσθαι τὸν...,καὶ μὴν ἐν σκηναῖς προστέτακται διαιτᾶσθαι τὸν...,None
7,feast_of_booths,In Flaccum,Section 116,urn:cts:greekLit:tlg0018.tlg030.1st1K-grc1:116,"Τοῦτο καινότατον ὑπέμεινε Φλάκκος ἐν χώρᾳ, ἧς ...","Τοῦτο καινότατον ὑπέμεινε Φλάκκος ἐν χώρᾳ, ἧς ...",None
8,feast_of_booths,De Plantatione,Section 31,urn:cts:greekLit:tlg0018.tlg010.1st1K-grc1:31,τὰ δὲ ἀμείνω καὶ τελειότερα τῷ μεσαιτάτῳ καὶ κ...,τὰ δὲ ἀμείνω καὶ τελειότερα τῷ μεσαιτάτῳ καὶ κ...,None
9,feast_of_booths,De Fuga Et Inventione,Section 153,urn:cts:greekLit:tlg0018.tlg017.1st1K-grc1:153,"ζητῶν οὖν 1 αὐτῶ H1 βούλετ’ H τῶν Mang.: τῶ G,...","ζητῶν οὖν 1 αὐτῶ H1 βούλετ’ H τῶν Mang.: τῶ G,...",None


# 11 - Focus on likely Hakhel contexts<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

*De Specialibus Legibus* is an obvious place to inspect because Philo discusses biblical laws and festivals there. This does not make the other works irrelevant; it merely gives us a manageable first set for close reading.

Review especially passages where several themes converge—for example, Booths plus seven, law plus hearing, or community/children vocabulary. The retrieved Greek must decide whether a hit concerns Hakhel or an unrelated use of the same word.

In [10]:
# Focus on likely Hakhel contexts
if evidence_df.empty:
    focused_df = evidence_df.copy()
else:
    focused_df = evidence_df[
        evidence_df["work"].fillna("").str.contains("Specialibus", case=False)
    ].sort_values(["urn", "theme"]).reset_index(drop=True)

print(f"Focused evidence rows: {len(focused_df)}")
focused_df[display_columns] if not focused_df.empty else focused_df

Focused evidence rows: 9


,theme,work,human_reference,urn,snippet,passage_text,error
0,seven,De Specialibus Legibus (lib. i‑iv),Book 1 Section 168,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:1.168,Ἐπεὶ δὲ τῶν θυσιῶν αἱ μέν εἰσιν ὑπὲρ ἅπαντος τ...,Ἐπεὶ δὲ τῶν θυσιῶν αἱ μέν εἰσιν ὑπὲρ ἅπαντος τ...,None
1,seven,De Specialibus Legibus (lib. i‑iv),Book 1 Section 188,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:1.188,τὰς μὲν οὖν ὡς ἑορτῆς θυσίας ἰσαρίθμους ἀπέφην...,τὰς μὲν οὖν ὡς ἑορτῆς θυσίας ἰσαρίθμους ἀπέφην...,None
2,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 204,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.204,Τελευταία δ’ ἐστὶν ἑορτὴ τῶν ἐνιαυσίων προσαγο...,Τελευταία δ’ ἐστὶν ἑορτὴ τῶν ἐνιαυσίων προσαγο...,None
3,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 206,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.206,καὶ μὴν ἐν σκηναῖς προστέτακται διαιτᾶσθαι τὸν...,καὶ μὴν ἐν σκηναῖς προστέτακται διαιτᾶσθαι τὸν...,None
4,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 207,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.207,"ὅτε μὲν γὰρ ἐν πεδίοις ἐστὶ τὰ τρέφοντα, μὴ συ...","ὅτε μὲν γὰρ ἐν πεδίοις ἐστὶ τὰ τρέφοντα, μὴ συ...",None
5,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 2 Section 250,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.250,κατὰ τὴν παλαιὰν ἐκείνην ἐξ Αἰγύπτου μετανάστα...,κατὰ τὴν παλαιὰν ἐκείνην ἐξ Αἰγύπτου μετανάστα...,None
6,seven,De Specialibus Legibus (lib. i‑iv),Book 2 Section 41,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.41,Περὶ τῶν δέκα έορτῶν. Εἰσὶ τοίνυν ἀριθμῷ δέκα ...,Περὶ τῶν δέκα έορτῶν. Εἰσὶ τοίνυν ἀριθμῷ δέκα ...,None
7,seven,De Specialibus Legibus (lib. i‑iv),Book 2 Section 58,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:2.58,Μωυσῆς δὲ ἀπὸ σεμνοτέρου πράγματος ἐκάλεσεν αὐ...,Μωυσῆς δὲ ἀπὸ σεμνοτέρου πράγματος ἐκάλεσεν αὐ...,None
8,feast_of_booths,De Specialibus Legibus (lib. i‑iv),Book 4 Section 129,urn:cts:greekLit:tlg0018.tlg024.1st1K-grc1:4.129,εἰκὸς μὲν οὖν τὸ τεράστιον τοῦ μεγαλουργηθέντο...,εἰκὸς μὲν οὖν τὸ τεράστιον τοῦ μεγαλουργηθέντο...,None


# 12 - Check the collected evidence<a class="anchor" id="bullet12"></a>
##### [Back to ToC](#TOC)

This small check makes retrieval failures and thematic coverage visible before we start interpreting the passages.

In [11]:
# Check the collected evidence
audit = {
    "mcp_server": mcp.name,
    "philo_textgroup": PHILO_TEXTGROUP,
    "search_terms": len(SEARCH_TERMS),
    "themed_hits": len(hits_df),
    "unique_passages": len(unique_urns),
    "retrieval_failures": 0 if passages_df.empty else int(passages_df["error"].notna().sum()),
    "focused_special_laws_rows": len(focused_df),
}
print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "mcp_server": "perseus",
  "philo_textgroup": "urn:cts:greekLit:tlg0018",
  "search_terms": 9,
  "themed_hits": 169,
  "unique_passages": 151,
  "retrieval_failures": 0,
  "focused_special_laws_rows": 9
}


# 13 - Export reproducible results<a class="anchor" id="bullet13"></a>
##### [Back to ToC](#TOC)

The CSV files retain CTS URNs so every interpretive note can be traced back to the retrieved source. The JSON file records the search plan and run summary.

In [12]:
# Export reproducible results
hits_path = OUTPUT_DIR / "philo_hakhel_search_hits.csv"
evidence_path = OUTPUT_DIR / "philo_hakhel_passage_evidence.csv"
focused_path = OUTPUT_DIR / "philo_hakhel_special_laws_focus.csv"
run_path = OUTPUT_DIR / "philo_hakhel_run.json"

hits_df.to_csv(hits_path, index=False, encoding="utf-8-sig")
evidence_df.to_csv(evidence_path, index=False, encoding="utf-8-sig")
focused_df.to_csv(focused_path, index=False, encoding="utf-8-sig")
run_path.write_text(
    json.dumps(
        {
            "scope": scope_summary,
            "search_terms": [
                {"theme": theme, "query": query, "search_kind": kind}
                for theme, query, kind in SEARCH_TERMS
            ],
            "search_summary": search_summary,
            "audit": audit,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

for path in [hits_path, evidence_path, focused_path, run_path]:
    print(path)

philo_hakhel_perseus_results\philo_hakhel_search_hits.csv
philo_hakhel_perseus_results\philo_hakhel_passage_evidence.csv
philo_hakhel_perseus_results\philo_hakhel_special_laws_focus.csv
philo_hakhel_perseus_results\philo_hakhel_run.json


# 14 - Questions for close reading<a class="anchor" id="bullet14"></a>
##### [Back to ToC](#TOC)

The resuled evidence table can be used to examine the following:

1. Does Philo connect a seven-year cycle with a public reading or hearing of the law?
2. Does he place that activity at the Feast of Booths?
3. Who performs the reading, and who forms the audience?
4. Are children, descendants, women, resident outsiders, or the entire people named explicitly?
5. What educational, political, ethical, or covenantal purpose does Philo assign to collective hearing?
6. How closely does his presentation follow Deuteronomy 31:10–13, and where does it interpret or reshape it?

Record conclusions only with the relevant CTS URNs. If the keyword search misses the central passage, add terms derived from the Greek of confirmed neighboring passages and rerun the Perseus MCP searches.

# 15 - Required libraries<a class="anchor" id="bullet15"></a>
##### [Back to ToC](#TOC)

The scripts in this notebook require the following Python libraries to be installed in the environment:

    perseus-mcp
    pandas
    tqdm

You can install a missing library from within Jupyter Notebook using either `pip` or `pip3`.

# 16 - Notebook details<a class="anchor" id="bullet16"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.2</td></tr>
    <tr><td><strong>Date</strong></td><td>3 August 2026</td></tr>
  </table>
</div>